# AITA Moral Stimuli Extraction (Study 2 - SSL)

This notebook extracts and filters post-level data from the AITA SQLite database for use in synthetic social learning (SSL) annotation tasks. The goal is to isolate moral stimuli with appropriate length, clarity, and vote distributions, and output a clean CSV with fields useful for downstream analysis.

Filtering criteria are based on cognitive reading limits (~238 words/minute) and the SSL paper's focus on moral judgments. We also consider entropy and class ratios to ensure balance across labels.


## Inspecting the AITA SQLite Dataset

The AITA SQLite database contains approximately 31,000 posts and over 9 million top-level comments from the r/AmItheAsshole subreddit. Each submission describes a personal moral conflict, and commenters respond with standardized verdict labels such as YTA (You’re the A-hole) or NTA (Not the A-hole). These labels reflect informal moral judgments and form the backbone of the SSL study design.

This file is large (~8GB) and stored in `data/raw/`, so it is excluded from version control using `.gitignore`.

The schema includes two main tables:
- `submission`: Contains post-level metadata and full text (`selftext`)
- `comment`: Contains comment text, verdicts, and upvotes linked to `submission_id`

The first step is to confirm the structure and preview the available columns for filtering and extraction.


In [1]:
import sqlite3
import pandas as pd

# Connect to the SQLite database (relative to notebook location)
db_path = "../data/raw/AmItheAsshole.sqlite"
conn = sqlite3.connect(db_path)

# List all tables in the database
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Available tables:", tables['name'].tolist())

# Preview the structure of the 'submission' table
submission_preview = pd.read_sql_query("SELECT * FROM submission LIMIT 5;", conn)
print("Submission table columns:", submission_preview.columns.tolist())
submission_preview.head()


Available tables: ['submission', 'comment']
Submission table columns: ['id', 'submission_id', 'title', 'selftext', 'created_utc', 'permalink', 'score']


,id,submission_id,title,selftext,created_utc,permalink,score
0,1,xt1ksm,AITA Monthly Open Forum Spooktober 2022,#Keep things civil. Rules still apply.\n\n##Th...,1664646465,/r/AmItheAsshole/comments/xt1ksm/aita_monthly_...,592
1,2,yiplwk,AITA for asking my friend to move a picture of...,"\n\nMe (M32) and my wife, Dahlia (F28) lost ou...",1667251988,/r/AmItheAsshole/comments/yiplwk/aita_for_aski...,16582
2,3,yiv572,AITA for asking my husband to stay with me whi...,Throwaway my family knows my account. I'll get...,1667266450,/r/AmItheAsshole/comments/yiv572/aita_for_aski...,4079
3,4,yimgaf,AITA for telling my SIL to stop talking about ...,My (37M) wife (37F) is pregnant with our first...,1667245059,/r/AmItheAsshole/comments/yimgaf/aita_for_tell...,9728
4,5,yin7pf,"AITA for wanting to meet my ""daughter"" after g...",Long story short: in my (40f) twenties I had a...,1667246573,/r/AmItheAsshole/comments/yin7pf/aita_for_want...,6889


In [2]:
# Show comment table schema
comment_preview = pd.read_sql_query("SELECT * FROM comment LIMIT 5;", conn)
print("Comment table columns:", comment_preview.columns.tolist())
comment_preview.head()


Comment table columns: ['id', 'submission_id', 'message', 'comment_id', 'parent_id', 'created_utc', 'score']


,id,submission_id,message,comment_id,parent_id,created_utc,score
0,1,xt1ksm,Do people with two digits to their age really ...,irs5v1y,t3_xt1ksm,1665421334,59
1,2,xt1ksm,Lots of posts in the last 3-4 days about rando...,isdxgsq,t3_xt1ksm,1665813660,55
2,3,xt1ksm,Sometimes I think people are making up stories...,iryatl4,t3_xt1ksm,1665529128,40
3,4,xt1ksm,Saw it on FB but it's hilarious how threads wi...,is3i5i9,t3_xt1ksm,1665622760,33
4,5,xt1ksm,The OP: My MIL can be a bit petty sometimes\n\...,ituw9ym,t3_xt1ksm,1666793880,32


## Join Submissions with Verdict Counts

Each submission is joined with its top-level comments containing standardized verdict labels. These include YTA (You’re the A-hole), NTA (Not the A-hole), ESH (Everyone Sucks Here), NAH (No A-holes Here), and INFO (Not Enough Info). Labels reflect public assessments of interpersonal conflict and provide structured social judgments for SSL analysis.

Posts with fewer than 10 words or more than 476 words are excluded based on estimated cognitive reading limits. The 476-word ceiling is based on a meta-analysis of adult silent reading rates in English, which found an average of 238 words per minute for non-fiction texts ([Brysbaert, 2019](https://doi.org/10.1016/j.jml.2019.104047)). Each row includes the post body, timestamp, Reddit permalink, and aggregated verdict counts. This structure supports downstream filtering by vote ratio, entropy, and readability.


In [3]:
# SQL to join submissions with structured verdict counts and total top-level comments
query = """
SELECT
    s.submission_id,
    s.created_utc,
    s.title,
    s.selftext,
    s.permalink,
    s.score AS post_score,

    -- Total number of top-level comments (with or without verdicts)
    COUNT(DISTINCT c.comment_id) AS num_comments,

    -- Structured verdict counts (only comments starting with verdict labels)
    SUM(CASE WHEN LOWER(c.message) LIKE 'yta%' THEN 1 ELSE 0 END) AS num_yta,
    SUM(CASE WHEN LOWER(c.message) LIKE 'nta%' THEN 1 ELSE 0 END) AS num_nta,
    SUM(CASE WHEN LOWER(c.message) LIKE 'esh%' THEN 1 ELSE 0 END) AS num_esh,
    SUM(CASE WHEN LOWER(c.message) LIKE 'nah%' THEN 1 ELSE 0 END) AS num_nah,
    SUM(CASE WHEN LOWER(c.message) LIKE 'info%' THEN 1 ELSE 0 END) AS num_info,

    -- Total number of structured verdicts across 5 types
    SUM(
        CASE WHEN
            LOWER(c.message) LIKE 'yta%' OR
            LOWER(c.message) LIKE 'nta%' OR
            LOWER(c.message) LIKE 'esh%' OR
            LOWER(c.message) LIKE 'nah%' OR
            LOWER(c.message) LIKE 'info%'
        THEN 1 ELSE 0 END
    ) AS verdict_total

FROM submission s
JOIN comment c ON SUBSTR(c.parent_id, 4) = s.submission_id
WHERE LENGTH(s.selftext) > 0
GROUP BY s.submission_id
"""

df = pd.read_sql_query(query, conn)

# Compute word count and apply reading time filter (10 < words ≤ 476)
df["word_count"] = df["selftext"].fillna("").apply(lambda x: len(x.split()))
df_filtered = df[(df["word_count"] > 10) & (df["word_count"] <= 476)].copy()

# Preview filtered sample
df_filtered[[
    "submission_id", "word_count", "num_comments", "verdict_total", "num_yta", "num_nta"
]].sample(min(3, len(df_filtered)))


,submission_id,word_count,num_comments,verdict_total,num_yta,num_nta
46,100awr1,179,68,50,48,0
21876,16wyrs0,316,24,16,0,16
1783,10hr4mg,166,34,24,1,12


## Flesch-Kincaid Readability Filter

The Flesch-Kincaid Grade Level metric estimates U.S. school-grade readability based on sentence length and word complexity. Posts above grade 8 are excluded to ensure clarity and ease of comprehension. This threshold aligns with communication standards for general adult audiences in the U.S. and supports efficient reading within a two-minute window.

In [4]:
import textstat

# Estimate U.S. grade-level readability using Flesch-Kincaid
df_filtered["fk_grade"] = df_filtered["selftext"].apply(lambda x: textstat.flesch_kincaid_grade(x))

# Filter for high readability: keep only posts at or below 8th-grade level
# Grade 8 is a reasonable threshold for general comprehension among U.S. adults,
# balancing accessibility with natural complexity. Common in public health and policy communication.
df_readable = df_filtered[df_filtered["fk_grade"] <= 8].copy()

# Preview a few results
df_readable[["submission_id", "word_count", "fk_grade", "num_yta", "num_nta"]].sample(3)


,submission_id,word_count,fk_grade,num_yta,num_nta
9720,1310e1m,151,7.087806,29,3
12920,141fkpi,363,6.350785,5,42
193,101i8n5,343,4.673546,0,54


## Verdict Ratios and Entropy

Moral stimuli with diverse or contested judgments provide stronger signals for studying synthetic social learning. Posts that show class disagreement or mixed moral framing are more informative than unanimous threads.

YTA to NTA ratios highlight where judgments are skewed or balanced, while entropy measures overall verdict diversity across five core types. High-entropy posts reflect varied moral reasoning and are prioritized for inclusion. Low-entropy posts, where verdicts are nearly unanimous, are less useful for modeling contested social knowledge.


In [5]:
import numpy as np

# Total structured verdicts across 5 canonical types
df_readable["verdict_total"] = (
    df_readable["num_yta"]
    + df_readable["num_nta"]
    + df_readable["num_esh"]
    + df_readable["num_nah"]
    + df_readable["num_info"]
)

# Remove rows with zero structured verdicts
df_readable = df_readable[df_readable["verdict_total"] > 0].copy()

# Compute YTA/NTA ratio
def compute_ratio(row):
    nta = row["num_nta"]
    yta = row["num_yta"]
    return yta / nta if nta > 0 else None

df_readable["yta_nta_ratio"] = df_readable.apply(compute_ratio, axis=1)

# Compute normalized entropy across verdict types
def verdict_entropy(row):
    counts = np.array([
        row["num_yta"],
        row["num_nta"],
        row["num_esh"],
        row["num_nah"],
        row["num_info"]
    ])
    total = counts.sum()
    if total == 0:
        return 0
    probs = counts / total
    return -np.sum([p * np.log2(p) for p in probs if p > 0])

df_readable["verdict_entropy"] = df_readable.apply(verdict_entropy, axis=1)

# Preview
df_readable[[
    "submission_id", "word_count", "verdict_total",
    "num_yta", "num_nta", "yta_nta_ratio", "verdict_entropy"
]].sample(3)


,submission_id,word_count,verdict_total,num_yta,num_nta,yta_nta_ratio,verdict_entropy
6393,11tre0m,341,246,3,240,0.0125,0.201016
5454,11je5od,235,275,262,0,NaN,0.274697
25837,yki4jk,432,18,0,16,0.0000,0.614369


## Filter for Balanced Moral Judgments

Posts selected for SSL annotation must show interpretable disagreement and meet key readability and engagement thresholds. Filtering focuses on judgment balance, diversity, and accessibility.

- YTA/NTA ratio in [0.25, 0.5, 0.75], with:
  - Number of comments (≥ 30)
  - Number of votes (post score available)
  - High readability (low Flesch-Kincaid score ≤ 8)

Entropy must be at least 1.0 to reflect disagreement across verdict types.


## Filtering Thresholds

These thresholds prioritize posts that reflect active disagreement and community engagement. Entropy ≥ 1.0 captures verdict diversity, while a minimum of 30 structured comments ensures verdict distributions are interpretable. These filters support the study of moral ambiguity and contested judgment within realistic social contexts.


In [6]:
# Final SSL filtering based on ratio, entropy, and count threshold
filtered_ssl = df_readable[
    (df_readable["yta_nta_ratio"].notnull()) &
    (df_readable["yta_nta_ratio"].between(0.25, 0.75)) &
    (df_readable["verdict_entropy"] >= 1.0) &
    (df_readable["verdict_total"] >= 30)
].copy()

print("SSL-qualified posts:", filtered_ssl.shape[0])

# Preview key fields
filtered_ssl[[
    "submission_id", "verdict_total", "yta_nta_ratio", "verdict_entropy"
]].sample(min(3, len(filtered_ssl)))


SSL-qualified posts: 762


,submission_id,verdict_total,yta_nta_ratio,verdict_entropy
28413,z85fyo,241,0.292035,1.701713
8880,12p2ahf,74,0.404762,1.489627
15026,14open3,37,0.666667,1.882865


## Export Final Stimuli for SSL Annotation

The filtered dataset includes 762 posts with structured verdict distributions that meet balance, readability, and disagreement thresholds. These posts reflect interpretable but contested moral judgments and are suitable for use in synthetic social learning tasks.

Each row includes the post body, structured verdict counts, word count, entropy, and YTA/NTA ratio. The dataset is saved as a CSV for downstream use in `study2` workflows.


In [7]:
# Reorder and export columns
export_cols = [
    "submission_id", "created_utc", "title", "selftext", "permalink",
    "post_score", "word_count", "verdict_total",
    "num_yta", "num_nta", "num_esh", "num_nah", "num_info",
    "yta_nta_ratio", "verdict_entropy"
]

filtered_ssl[export_cols].to_csv("../data/clean/aita_ssl_filtered.csv", index=False)
print("Exported to ../data/clean/aita_ssl_filtered.csv")


Exported to ../data/clean/aita_ssl_filtered.csv
